In [1]:
import os
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
import requests
import time
from Bio import SeqIO
from torch.utils.data import TensorDataset, DataLoader
import copy

import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

/opt/miniconda3/envs/AI/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
labels = pd.read_csv("G-HumanEssential.tsv", sep="\t")

entrez_ids = labels["Gene ID"].astype(str).tolist()

url = "https://mygene.info/v3/gene"

mapping = {}

batch_size = 1000

for start in range(0, len(entrez_ids), batch_size):
    batch = entrez_ids[start:start + batch_size]

    response = requests.post(
        url,
        data={
            "ids": ",".join(batch),
            "scopes": "entrezgene",
            "fields": "entrezgene,ensembl.gene,symbol",
            "species": "human"
        }
    )

    response.raise_for_status()

    results = response.json()

    for result in results:
        if result.get("notfound"):
            continue

        entrez = str(result.get("entrezgene"))

        ensembl_data = result.get("ensembl")

        if isinstance(ensembl_data, dict):
            ensembl = ensembl_data.get("gene")

        elif isinstance(ensembl_data, list):
            ensembl = None

            for item in ensembl_data:
                if isinstance(item, dict) and item.get("gene"):
                    ensembl = item["gene"]
                    break

        else:
            ensembl = None

        if entrez and ensembl:
            mapping[entrez] = ensembl

    print(
        f"Processed {min(start + batch_size, len(entrez_ids)):,} "
        f"/ {len(entrez_ids):,} | "
        f"Mapped: {len(mapping):,}"
    )

    time.sleep(0.2)

print("\nFinished.")
print(f"Total labels: {len(entrez_ids):,}")
print(f"Successfully mapped: {len(mapping):,}")
print(f"Not mapped: {len(entrez_ids) - len(mapping):,}")

Processed 1,000 / 18,528 | Mapped: 992
Processed 2,000 / 18,528 | Mapped: 1,981
Processed 3,000 / 18,528 | Mapped: 2,975
Processed 4,000 / 18,528 | Mapped: 3,969
Processed 5,000 / 18,528 | Mapped: 4,957
Processed 6,000 / 18,528 | Mapped: 5,944
Processed 7,000 / 18,528 | Mapped: 6,932
Processed 8,000 / 18,528 | Mapped: 7,923
Processed 9,000 / 18,528 | Mapped: 8,916
Processed 10,000 / 18,528 | Mapped: 9,911
Processed 11,000 / 18,528 | Mapped: 10,901
Processed 12,000 / 18,528 | Mapped: 11,885
Processed 13,000 / 18,528 | Mapped: 12,879
Processed 14,000 / 18,528 | Mapped: 13,867
Processed 15,000 / 18,528 | Mapped: 14,856
Processed 16,000 / 18,528 | Mapped: 15,848
Processed 17,000 / 18,528 | Mapped: 16,838
Processed 18,000 / 18,528 | Mapped: 17,828
Processed 18,528 / 18,528 | Mapped: 18,352

Finished.
Total labels: 18,528
Successfully mapped: 18,352
Not mapped: 176


In [3]:
mapping_df = pd.DataFrame(
    list(mapping.items()),
    columns=["Gene ID", "Ensembl Gene ID"]
)

mapping_df["Gene ID"] = mapping_df["Gene ID"].astype(str)

labels_clean = labels.copy()
labels_clean["Gene ID"] = labels_clean["Gene ID"].astype(str)

labeled_genes = labels_clean.merge(
    mapping_df,
    on="Gene ID",
    how="inner"
)

print("Labeled genes:", len(labeled_genes))
print("\nClass distribution:")
print(labeled_genes[
    "Essentiality (determined from multiple datasets)"
].value_counts())

Labeled genes: 18351

Class distribution:
Essentiality (determined from multiple datasets)
Non-essential    16841
Essential         1510
Name: count, dtype: int64


In [ ]:
def parse_header_ids(header, record_id):
    gene_id = None
    transcript_id = None

    for field in header.split():
        if field.startswith("gene:"):
            gene_id = field.split(":", 1)[1].split(".", 1)[0]
        elif field.startswith("transcript:"):
            transcript_id = field.split(":", 1)[1].split(".", 1)[0]

    if transcript_id is None and str(record_id).startswith("ENST"):
        transcript_id = str(record_id).split(".", 1)[0]

    return gene_id, transcript_id


fasta_path = "Homo_sapiens.GRCh38.cds.all.fa"

target_ensembl_ids = set(
    labeled_genes["Ensembl Gene ID"].astype(str)
)

cds_by_transcript = {}

for record in SeqIO.parse(fasta_path, "fasta"):
    gene_id, transcript_id = parse_header_ids(record.description, record.id)

    if gene_id not in target_ensembl_ids or transcript_id is None:
        continue

    sequence = str(record.seq)

    if (
        transcript_id not in cds_by_transcript
        or len(sequence) > len(cds_by_transcript[transcript_id][1])
    ):
        cds_by_transcript[transcript_id] = (gene_id, sequence)

print(f"Target genes:              {len(target_ensembl_ids):,}")
print(f"CDS transcripts retained:  {len(cds_by_transcript):,}")
print(f"Genes with at least one CDS: {len({g for g, _ in cds_by_transcript.values()}):,}")

Target genes:              18,349
CDS transcripts retained:  102,751
Genes with at least one CDS: 18,013


In [ ]:
protein_fasta_path = "Homo_sapiens.GRCh38.pep.all.fa"

pep_by_transcript = {}

for record in SeqIO.parse(protein_fasta_path, "fasta"):
    gene_id, transcript_id = parse_header_ids(record.description, record.id)

    if gene_id not in target_ensembl_ids or transcript_id is None:
        continue

    sequence = str(record.seq)

    if (
        transcript_id not in pep_by_transcript
        or len(sequence) > len(pep_by_transcript[transcript_id][1])
    ):
        pep_by_transcript[transcript_id] = (gene_id, sequence)

print(f"Target genes:                   {len(target_ensembl_ids):,}")
print(f"Protein transcripts retained:   {len(pep_by_transcript):,}")
print(f"Genes with at least one protein: {len({g for g, _ in pep_by_transcript.values()}):,}")

Target genes:                   18,349
Protein transcripts retained:   102,751
Genes with at least one protein: 18,013


In [ ]:
label_column = "Essentiality (determined from multiple datasets)"

paired = {}

shared_transcripts = set(cds_by_transcript) & set(pep_by_transcript)

for transcript_id in shared_transcripts:
    gene_id, dna_seq = cds_by_transcript[transcript_id]
    pep_gene_id, protein_seq = pep_by_transcript[transcript_id]

    if gene_id != pep_gene_id:
        continue

    if gene_id not in paired or len(protein_seq) > len(paired[gene_id][1]):
        paired[gene_id] = (dna_seq, protein_seq, transcript_id)

sequence_df = pd.DataFrame(
    [
        {
            "Ensembl Gene ID": gene_id,
            "Ensembl Transcript ID": transcript_id,
            "dna_sequence": dna_seq,
            "protein_sequence": protein_seq,
        }
        for gene_id, (dna_seq, protein_seq, transcript_id) in paired.items()
    ]
)

dataset = labeled_genes.merge(
    sequence_df,
    on="Ensembl Gene ID",
    how="inner"
)

dataset["label"] = (
    dataset[label_column] == "Essential"
).astype(int)

dataset = dataset[
    [
        "Gene ID",
        "Ensembl Gene ID",
        "Ensembl Transcript ID",
        "dna_sequence",
        "protein_sequence",
        "label",
    ]
]

print(f"Shared transcripts: {len(shared_transcripts):,}")
print(f"Final dataset size (DNA + protein, same transcript): {len(dataset):,}")
print(dataset["label"].value_counts())


Shared transcripts: 102,751
Final dataset size (DNA + protein, same transcript): 18,015
label
0    16506
1     1509
Name: count, dtype: int64


In [ ]:
NTV3_REPO = "InstaDeepAI/NTv3_100M_pre"
MAX_SEQ_LEN = 4096  
EMBED_CACHE_PATH = "ntv3_embeddings.npy"

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)

ntv3_tokenizer = AutoTokenizer.from_pretrained(NTV3_REPO, trust_remote_code=True)
ntv3_model = AutoModelForMaskedLM.from_pretrained(NTV3_REPO, trust_remote_code=True)
ntv3_model = ntv3_model.to(device)
ntv3_model.eval()

EMBED_BATCH_SIZE = 2 if device.type == "cpu" else 8

print("Loaded", NTV3_REPO)
print("Max sequence length:", MAX_SEQ_LEN)
print("Embedding batch size:", EMBED_BATCH_SIZE)

Device: mps


Loading weights: 100%|██████████| 207/207 [00:00<00:00, 7629.36it/s]


Loaded InstaDeepAI/NTv3_100M_pre
Max sequence length: 4096
Embedding batch size: 8


In [8]:
def sanitize_dna(sequence):
    seq = sequence.upper().replace("U", "T")
    seq = "".join(base if base in "ACGTN" else "N" for base in seq)
    return seq if seq else "N"


def chunk_sequence(sequence, max_len=MAX_SEQ_LEN):
    return [
        sequence[i:i + max_len]
        for i in range(0, len(sequence), max_len)
    ]


@torch.no_grad()
def embed_chunk_batch(chunks):
    lengths = [len(chunk) for chunk in chunks]
    padded_len = ((max(lengths) + 127) // 128) * 128
    padded = [chunk + "N" * (padded_len - len(chunk)) for chunk in chunks]

    encoded = ntv3_tokenizer(
        padded,
        add_special_tokens=False,
        padding=False,
        return_tensors="pt",
    )

    input_ids = encoded["input_ids"].to(device)
    model_mask = torch.ones_like(input_ids)

    pooling_mask = torch.zeros(
        len(chunks), padded_len, dtype=torch.float32, device=device
    )
    for i, length in enumerate(lengths):
        pooling_mask[i, :length] = 1.0

    hidden = ntv3_model(
        input_ids=input_ids,
        attention_mask=model_mask,
        output_hidden_states=True,
    ).hidden_states[-1]

    pooled = (hidden * pooling_mask.unsqueeze(-1)).sum(dim=1)
    pooled = pooled / pooling_mask.sum(dim=1, keepdim=True).clamp(min=1.0)

    return pooled.cpu().numpy()


y = dataset["label"].values

X_ntv3 = None
if os.path.exists(EMBED_CACHE_PATH):
    cached = np.load(EMBED_CACHE_PATH)
    if cached.shape[0] == len(dataset):
        X_ntv3 = cached
        print("Loaded cached embeddings from", EMBED_CACHE_PATH)
    else:
        print(
            f"NTv3 cache has {cached.shape[0]} rows, "
            f"dataset has {len(dataset)}; recomputing."
        )

if X_ntv3 is None:
    all_chunks = []
    chunk_owners = []

    for i, seq in enumerate(dataset["dna_sequence"]):
        chunks = chunk_sequence(sanitize_dna(seq))
        all_chunks.extend(chunks)
        chunk_owners.extend([i] * len(chunks))

    print(f"Sequences: {len(dataset):,}")
    print(f"Chunks to embed: {len(all_chunks):,}")

    chunk_embeddings = []

    for start in tqdm(range(0, len(all_chunks), EMBED_BATCH_SIZE), desc="NTv3 embeddings"):
        batch_chunks = all_chunks[start:start + EMBED_BATCH_SIZE]
        chunk_embeddings.append(embed_chunk_batch(batch_chunks))

    chunk_embeddings = np.vstack(chunk_embeddings)

    X_ntv3 = np.zeros(
        (len(dataset), chunk_embeddings.shape[1]),
        dtype=np.float32,
    )
    counts = np.zeros(len(dataset), dtype=np.float32)

    for owner, embedding in zip(chunk_owners, chunk_embeddings):
        X_ntv3[owner] += embedding
        counts[owner] += 1

    X_ntv3 /= counts[:, None]
    np.save(EMBED_CACHE_PATH, X_ntv3)
    print("Saved embeddings to", EMBED_CACHE_PATH)

print("X_ntv3 shape:", X_ntv3.shape)
print("y shape:", y.shape)

Sequences: 18,015
Chunks to embed: 19,509


NTv3 embeddings: 100%|██████████| 2439/2439 [53:41<00:00,  1.32s/it] 

Saved embeddings to ntv3_embeddings.npy
X_ntv3 shape: (18015, 768)
y shape: (18015,)


In [ ]:
if "ntv3_model" in globals():
    del ntv3_model
if "ntv3_tokenizer" in globals():
    del ntv3_tokenizer
gc.collect()

if device.type == "cuda":
    torch.cuda.empty_cache()
elif device.type == "mps":
    torch.mps.empty_cache()

ESM2_REPO = "facebook/esm2_t33_650M_UR50D"
ESM2_MAX_LEN = 1022  
ESM2_CACHE_PATH = "esm2_embeddings.npy"
ESM2_BATCH_SIZE = 1 if device.type == "cpu" else (2 if device.type == "mps" else 8)

esm2_tokenizer = AutoTokenizer.from_pretrained(ESM2_REPO)
esm2_model = AutoModel.from_pretrained(ESM2_REPO)
esm2_model = esm2_model.to(device)
esm2_model.eval()

print("Loaded", ESM2_REPO)
print("Hidden size:", esm2_model.config.hidden_size)
print("Max protein length:", ESM2_MAX_LEN)
print("Embedding batch size:", ESM2_BATCH_SIZE)

Loading weights: 100%|██████████| 534/534 [00:00<00:00, 8828.62it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded facebook/esm2_t33_650M_UR50D
Hidden size: 1280
Max protein length: 1022
Embedding batch size: 2


In [10]:
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")


def sanitize_protein(sequence):
    seq = sequence.upper().replace("*", "")
    seq = "".join(aa if aa in STANDARD_AA else "X" for aa in seq)
    return seq if seq else "X"


def chunk_protein(sequence, max_len=ESM2_MAX_LEN):
    return [
        sequence[i:i + max_len]
        for i in range(0, len(sequence), max_len)
    ]


@torch.no_grad()
def embed_protein_chunk_batch(chunks):
    encoded = esm2_tokenizer(
        chunks,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=ESM2_MAX_LEN + 2,
        add_special_tokens=True,
    )
    encoded = {key: value.to(device) for key, value in encoded.items()}

    hidden = esm2_model(**encoded).last_hidden_state
    attention_mask = encoded["attention_mask"].float()

    token_mask = attention_mask.clone()
    token_mask[:, 0] = 0
    seq_lens = attention_mask.sum(dim=1).long()
    token_mask[
        torch.arange(token_mask.size(0), device=device),
        seq_lens - 1,
    ] = 0

    pooled = (hidden * token_mask.unsqueeze(-1)).sum(dim=1)
    pooled = pooled / token_mask.sum(dim=1, keepdim=True).clamp(min=1.0)

    return pooled.cpu().numpy()


X_esm2 = None
if os.path.exists(ESM2_CACHE_PATH):
    cached = np.load(ESM2_CACHE_PATH)
    if cached.shape[0] == len(dataset):
        X_esm2 = cached
        print("Loaded cached embeddings from", ESM2_CACHE_PATH)
    else:
        print(
            f"ESM2 cache has {cached.shape[0]} rows, "
            f"dataset has {len(dataset)}; recomputing."
        )

if X_esm2 is None:
    all_chunks = []
    chunk_owners = []

    for i, seq in enumerate(dataset["protein_sequence"]):
        chunks = chunk_protein(sanitize_protein(seq))
        all_chunks.extend(chunks)
        chunk_owners.extend([i] * len(chunks))

    print(f"Proteins: {len(dataset):,}")
    print(f"Chunks to embed: {len(all_chunks):,}")

    chunk_embeddings = []

    for start in tqdm(range(0, len(all_chunks), ESM2_BATCH_SIZE), desc="ESM2 embeddings"):
        batch_chunks = all_chunks[start:start + ESM2_BATCH_SIZE]
        chunk_embeddings.append(embed_protein_chunk_batch(batch_chunks))

    chunk_embeddings = np.vstack(chunk_embeddings)

    X_esm2 = np.zeros(
        (len(dataset), chunk_embeddings.shape[1]),
        dtype=np.float32,
    )
    counts = np.zeros(len(dataset), dtype=np.float32)

    for owner, embedding in zip(chunk_owners, chunk_embeddings):
        X_esm2[owner] += embedding
        counts[owner] += 1

    X_esm2 /= counts[:, None]
    np.save(ESM2_CACHE_PATH, X_esm2)
    print("Saved embeddings to", ESM2_CACHE_PATH)

print("X_esm2 shape:", X_esm2.shape)

Proteins: 18,015
Chunks to embed: 20,990


ESM2 embeddings:  74%|███████▍  | 7807/10495 [12:58:53<1:10:39,  1.58s/it]   2026-09-10 09:05:01.582 python[7491:6035299] Error creating directory 
 The volume ‚ÄúMacintosh HD‚Äù is out of space. You can‚Äôt save the file ‚Äúmpsgraph-7491-2026-09-10_09_05_01-2952841200‚Äù because the volume ‚ÄúMacintosh HD‚Äù is out of space.
ESM2 embeddings: 100%|██████████| 10495/10495 [20:45:28<00:00,  7.12s/it]    


Saved embeddings to esm2_embeddings.npy
X_esm2 shape: (18015, 1280)


In [ ]:
if "esm2_model" in globals():
    del esm2_model
if "esm2_tokenizer" in globals():
    del esm2_tokenizer
gc.collect()

if device.type == "cuda":
    torch.cuda.empty_cache()
elif device.type == "mps":
    torch.mps.empty_cache()

if X_ntv3.shape[0] != X_esm2.shape[0]:
    raise ValueError(
        f"Embedding row mismatch: NTv3 {X_ntv3.shape[0]} vs ESM2 {X_esm2.shape[0]}"
    )

X = np.concatenate([X_ntv3, X_esm2], axis=1).astype(np.float32)

print("X_ntv3 (DNA) shape:     ", X_ntv3.shape)
print("X_esm2 (protein) shape: ", X_esm2.shape)
print("X concat shape:         ", X.shape)
print("y shape:                ", y.shape)

X_ntv3 (DNA) shape:      (18015, 768)
X_esm2 (protein) shape:  (18015, 1280)
X concat shape:          (18015, 2048)
y shape:                 (18015,)


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining distribution:")
print(pd.Series(y_train).value_counts())

print("\nTesting distribution:")
print(pd.Series(y_test).value_counts())

Training samples: 14412
Testing samples: 3603

Training distribution:
0    13205
1     1207
Name: count, dtype: int64

Testing distribution:
0    3301
1     302
Name: count, dtype: int64


In [17]:
torch.manual_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    random_state=42,
    stratify=y_train,
)

torch_scaler = StandardScaler()
X_tr_scaled = torch_scaler.fit_transform(X_tr)
X_val_scaled = torch_scaler.transform(X_val)
X_test_scaled_torch = torch_scaler.transform(X_test)

train_ds = TensorDataset(
    torch.tensor(X_tr_scaled, dtype=torch.float32),
    torch.tensor(y_tr, dtype=torch.long),
)

train_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
)

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32, device=device)
X_test_tensor = torch.tensor(X_test_scaled_torch, dtype=torch.float32, device=device)


class EssentialityMLP(nn.Module):
    def __init__(self, in_features, n_classes=2, dropout=0.3):
        super().__init__()

        self.fc1 = nn.Linear(in_features, 4096)
        self.fc2 = nn.Linear(4096, 1024)
        self.fc3 = nn.Linear(1024, 512)
        self.fc4 = nn.Linear(512, n_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.dropout(self.relu(self.fc3(x)))
        return self.fc4(x) 


model = EssentialityMLP(in_features=X_tr.shape[1]).to(device)

print(model)
print("Input features (NTv3 DNA + ESM2 protein):", X_tr.shape[1])
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

n_pos = int((y_tr == 1).sum())
n_neg = int((y_tr == 0).sum())

class_weights = torch.tensor(
    [len(y_tr) / (2 * n_neg), len(y_tr) / (2 * n_pos)],
    dtype=torch.float32,
    device=device,
)

print(f"\nTraining rows: {len(y_tr):,} (essential {n_pos:,} / non-essential {n_neg:,})")
print(f"Validation rows: {len(y_val):,} (essential {int(y_val.sum()):,})")
print(f"Class weights: non-essential {class_weights[0]:.3f} | essential {class_weights[1]:.3f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)

MAX_EPOCHS = 300
PATIENCE = 30

best_ap = -np.inf
best_state = None
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()

    epoch_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()

        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(yb)

    epoch_loss /= len(train_ds)

    model.eval()

    with torch.no_grad():
        val_prob = torch.softmax(model(X_val_tensor), dim=1)[:, 1].cpu().numpy()

    val_ap = average_precision_score(y_val, val_prob)

    if val_ap > best_ap:
        best_ap = val_ap
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 10 == 0:
        print(f"epoch {epoch:3d} | train loss {epoch_loss:.4f} | val PR-AUC {val_ap:.4f}")

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

model.load_state_dict(best_state)

print(f"\nBest epoch: {best_epoch}")
print(f"Validation PR-AUC: {best_ap:.4f} (baseline {y_val.mean():.4f})")

Device: mps
EssentialityMLP(
  (fc1): Linear(in_features=2048, out_features=4096, bias=True)
  (fc2): Linear(in_features=4096, out_features=1024, bias=True)
  (fc3): Linear(in_features=1024, out_features=512, bias=True)
  (fc4): Linear(in_features=512, out_features=2, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
)
Input features (NTv3 DNA + ESM2 protein): 2048
Trainable parameters: 13113858

Training rows: 12,250 (essential 1,026 / non-essential 11,224)
Validation rows: 2,162 (essential 181)
Class weights: non-essential 0.546 | essential 5.970
epoch   1 | train loss 0.5585 | val PR-AUC 0.4621
epoch  10 | train loss 0.3655 | val PR-AUC 0.4230
epoch  20 | train loss 0.3666 | val PR-AUC 0.4273
epoch  30 | train loss 0.2813 | val PR-AUC 0.3992

Early stopping at epoch 36.

Best epoch: 6
Validation PR-AUC: 0.5203 (baseline 0.0837)


In [18]:
model.eval()

with torch.no_grad():
    test_prob = torch.softmax(model(X_test_tensor), dim=1)

    test_pred = test_prob.argmax(dim=1).cpu().numpy()
    test_prob_essential = test_prob[:, 1].cpu().numpy()

print("Classification report (softmax argmax):\n")
print(classification_report(y_test, test_pred, target_names=["Non-essential", "Essential"]))

print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred))

print(f"\nROC-AUC: {roc_auc_score(y_test, test_prob_essential):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, test_prob_essential):.4f}")
print(f"PR-AUC baseline (prevalence): {y_test.mean():.4f}")

Classification report (softmax argmax):

               precision    recall  f1-score   support

Non-essential       0.98      0.84      0.91      3301
    Essential       0.32      0.82      0.46       302

     accuracy                           0.84      3603
    macro avg       0.65      0.83      0.68      3603
 weighted avg       0.93      0.84      0.87      3603

Confusion matrix:
[[2783  518]
 [  55  247]]

ROC-AUC: 0.9059
PR-AUC:  0.5386
PR-AUC baseline (prevalence): 0.0838
